In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Import your project files
from lib.networks import EMCADNet
from utils.dataloader import generate_hupanno

# ===========================================================================
# Configuration
# ===========================================================================
IMAGE_DIR = '/kaggle/working/data/Kvasir/test/images/'  # Change to your test image path
MASK_DIR  = '/kaggle/working/data/Kvasir/test/masks/'   # Change to your test mask path
MODEL_PATH = './model_pth_hup/ws_hupanno-best.pth'
OUTPUT_FILE = 'hupanno_visual_proof.png'
NUM_IMAGES = 4  # Number of random images to visualize

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===========================================================================
# Helper Functions
# ===========================================================================
def load_image_and_gt(img_path, gt_path, target_size=352):
    img = np.array(Image.open(img_path).convert('RGB'))
    gt  = np.array(Image.open(gt_path).convert('L'))
    
    # Resize
    img_resized = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    gt_resized  = cv2.resize(gt,  (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    
    return img_resized, (gt_resized > 127).astype(np.float32)

def draw_weak_annotations(img, y_in, y_out, lrp_fg, lrp_bg, lrp_uncertain):
    """Draws the weak polygons on the image to show what the model is trained on."""
    vis = img.copy()
    
    # 1. Draw P_in (Certain Foreground) - Solid Green Fill
    y_in_u8 = (y_in * 255).astype(np.uint8)
    contours_in, _ = cv2.findContours(y_in_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours_in:
        cv2.drawContours(vis, contours_in, -1, (0, 255, 0), -1) # Green fill
        cv2.drawContours(vis, contours_in, -1, (0, 200, 0), 2)  # Dark green border

    # 2. Draw P_out (Outer Envelope) - Red Outline
    y_out_u8 = (y_out * 255).astype(np.uint8)
    contours_out, _ = cv2.findContours(y_out_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours_out:
        cv2.drawContours(vis, contours_out, -1, (255, 0, 0), 2) # Red outline

    # 3. Draw LRP Patches (Local Refinement) - Bright Yellow/Cyan Outlines
    lrp_mask = (lrp_fg + lrp_bg + lrp_uncertain > 0).astype(np.uint8) * 255
    contours_lrp, _ = cv2.findContours(lrp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours_lrp:
        cv2.drawContours(vis, contours_lrp, -1, (0, 255, 255), 2) # Cyan/Yellow outline

    # Blend with original image (50% opacity for the annotations)
    vis = cv2.addWeighted(img, 0.5, vis, 0.5, 0)
    return vis

# ===========================================================================
# Main Execution
# ===========================================================================
if __name__ == '__main__':
    print("Loading model...")
    model = EMCADNet(num_classes=1, encoder='pvt_v2_b2', pretrain=False).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    # Get random image files
    img_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))])
    np.random.seed(42)
    selected_files = np.random.choice(img_files, NUM_IMAGES, replace=False)

    fig, axes = plt.subplots(4, NUM_IMAGES, figsize=(20, 20))
    fig.suptitle('PROOF OF WEAK SUPERVISION: What the model sees vs. What it predicts', 
                 fontsize=24, fontweight='bold', y=0.98)

    for col, fname in enumerate(selected_files):
        img_path = os.path.join(IMAGE_DIR, fname)
        gt_path  = os.path.join(MASK_DIR, fname.replace('.jpg', '.png'))
        
        # 1. Load Data
        img, gt = load_image_and_gt(img_path, gt_path)
        gt_np = gt.astype(np.float32)
        
        # 2. Generate Weak Annotations (This is ALL the model sees during training)
        (y_in, y_out, omega_delta, lrp_fg, lrp_bg, 
         lrp_uncertain, lrp_mask, y_c) = generate_hupanno(gt_np, K=2)
         
        # 3. Run Model Inference (Model only gets the raw image)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        # Normalize
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_tensor = (img_tensor - mean) / std
        
        with torch.no_grad():
            preds = model(img_tensor.unsqueeze(0).to(DEVICE), mode='test')
            pred_prob = torch.sigmoid(preds[-1]).squeeze().cpu().numpy()
            pred_bin = (pred_prob > 0.5).astype(np.float32)

        # --- PLOTTING ---
        
        # Row 1: Original Image
        axes[0, col].imshow(img)
        axes[0, col].set_title('Original Image', fontsize=16, fontweight='bold')
        axes[0, col].axis('off')
        
        # Row 2: Weak Annotations (The "Supervision")
        vis_weak = draw_weak_annotations(img, y_in, y_out, lrp_fg, lrp_bg, lrp_uncertain)
        axes[1, col].imshow(vis_weak)
        axes[1, col].set_title('Weak Annotations (Training Input)\nGreen=P_in, Red=P_out, Cyan=LRP', 
                               fontsize=14, fontweight='bold', color='blue')
        axes[1, col].axis('off')
        
        # Row 3: Model Prediction
        axes[2, col].imshow(pred_bin, cmap='gray')
        axes[2, col].set_title('Model Prediction (Dense Output)', 
                               fontsize=16, fontweight='bold', color='green')
        axes[2, col].axis('off')
        
        # Row 4: Ground Truth
        axes[3, col].imshow(gt, cmap='gray')
        axes[3, col].set_title('Ground Truth (Never seen in training)', 
                               fontsize=16, fontweight='bold')
        axes[3, col].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches='tight')
    print(f"\n✅ Visualization saved to {OUTPUT_FILE}")
    print("Look at Row 2 (coarse shapes) vs Row 3 (dense mask). The model learned the dense boundary purely from the coarse shapes!")